In [1]:
import numpy as np
import pandas as pd

from scipy.stats import chisquare, norm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (
    proportion_effectsize,
    proportions_ztest,
)
from IPython.display import display

SEED = 42
ALPHA = 0.05
TARGET_POWER = 0.80
BASELINE_RATE = 0.01
MIN_RELATIVE_LIFT = 0.20
SRM_THRESHOLD = 0.001

TARGET_RATE = BASELINE_RATE * (1 + MIN_RELATIVE_LIFT)

print(f"Hypothetical baseline: {BASELINE_RATE:.2%}")
print(f"Target conversion: {TARGET_RATE:.2%}")
print(
    f"Minimum detectable difference: "
    f"{(TARGET_RATE - BASELINE_RATE) * 100:.2f} percentage points"
)

Hypothetical baseline: 1.00%
Target conversion: 1.20%
Minimum detectable difference: 0.20 percentage points


In [2]:
effect_size = abs(
    proportion_effectsize(TARGET_RATE, BASELINE_RATE)
)

required_per_group = int(
    np.ceil(
        NormalIndPower().solve_power(
            effect_size=effect_size,
            alpha=ALPHA,
            power=TARGET_POWER,
            ratio=1,
            alternative="two-sided",
        )
    )
)

total_visitors = 2 * required_per_group

display(pd.DataFrame({
    "planned_visitors_per_group": [required_per_group],
    "planned_total_visitors": [total_visitors],
    "target_power": [TARGET_POWER],
    "relative_mde_pct": [MIN_RELATIVE_LIFT * 100],
}))

,planned_visitors_per_group,planned_total_visitors,target_power,relative_mde_pct
0,42607,85214,0.8,20.0


In [3]:
rng = np.random.default_rng(SEED)

# Independent 50/50 assignment for each synthetic visitor.
groups = np.where(
    rng.random(total_visitors) < 0.5,
    "control",
    "treatment",
)

purchase_probabilities = np.where(
    groups == "control",
    BASELINE_RATE,
    TARGET_RATE,
)

experiment = pd.DataFrame({
    "visitor_id": np.arange(1, total_visitors + 1),
    "group": groups,
    "purchased_within_7_days": rng.binomial(
        n=1,
        p=purchase_probabilities,
    ),
})

# Every synthetic visitor has a fully observed seven-day window.
experiment["outcome_window_complete"] = True

assert experiment["visitor_id"].is_unique
assert experiment["group"].isin(["control", "treatment"]).all()
assert experiment["purchased_within_7_days"].isin([0, 1]).all()
assert experiment["outcome_window_complete"].all()

display(experiment.head())

,visitor_id,group,purchased_within_7_days,outcome_window_complete
0,1,treatment,0,True
1,2,control,0,True
2,3,treatment,0,True
3,4,treatment,0,True
4,5,control,0,True


In [4]:
group_sizes = (
    experiment["group"]
    .value_counts()
    .reindex(["control", "treatment"], fill_value=0)
)

expected_sizes = np.array([
    total_visitors / 2,
    total_visitors / 2,
])

srm_statistic, srm_p_value = chisquare(
    f_obs=group_sizes.to_numpy(),
    f_exp=expected_sizes,
)

srm_passed = bool(srm_p_value >= SRM_THRESHOLD)

display(pd.DataFrame({
    "group": group_sizes.index,
    "assigned_visitors": group_sizes.values,
    "expected_visitors": expected_sizes,
}))

print(f"SRM p-value: {srm_p_value:.6f}")
print(f"SRM check passed: {srm_passed}")

,group,assigned_visitors,expected_visitors
0,control,42353,42607.0
1,treatment,42861,42607.0


SRM p-value: 0.081818
SRM check passed: True


In [5]:
group_summary = (
    experiment.groupby("group")
    .agg(
        assigned_visitors=("visitor_id", "size"),
        purchasing_visitors=("purchased_within_7_days", "sum"),
    )
    .reindex(["control", "treatment"])
)

group_summary["conversion_rate"] = (
    group_summary["purchasing_visitors"]
    / group_summary["assigned_visitors"]
)

group_summary["conversion_pct"] = (
    group_summary["conversion_rate"] * 100
)

display(group_summary.round(4))

counts = group_summary["purchasing_visitors"].to_numpy()
sizes = group_summary["assigned_visitors"].to_numpy()
rates = counts / sizes

# Check that the large-sample approximation is reasonable.
assert (counts >= 10).all()
assert ((sizes - counts) >= 10).all()

z_statistic, p_value = proportions_ztest(
    count=counts,
    nobs=sizes,
    alternative="two-sided",
)

difference = rates[1] - rates[0]

difference_se = np.sqrt(
    rates[0] * (1 - rates[0]) / sizes[0]
    + rates[1] * (1 - rates[1]) / sizes[1]
)

critical_value = norm.ppf(1 - ALPHA / 2)

lower_difference = difference - critical_value * difference_se
upper_difference = difference + critical_value * difference_se

relative_lift = (
    difference / rates[0] if rates[0] > 0 else np.nan
)

results = pd.DataFrame({
    "control_conversion_pct": [rates[0] * 100],
    "treatment_conversion_pct": [rates[1] * 100],
    "absolute_lift_pp": [difference * 100],
    "relative_lift_pct": [relative_lift * 100],
    "difference_lower_95_pp": [lower_difference * 100],
    "difference_upper_95_pp": [upper_difference * 100],
    "two_sided_p_value": [p_value],
})

display(results)

,assigned_visitors,purchasing_visitors,conversion_rate,conversion_pct
group,,,,
control,42353,410,0.0097,0.9681
treatment,42861,506,0.0118,1.1806


,control_conversion_pct,treatment_conversion_pct,absolute_lift_pp,relative_lift_pct,difference_lower_95_pp,difference_upper_95_pp,two_sided_p_value
0,0.968054,1.18056,0.212506,21.951891,0.074118,0.350895,0.002632


In [6]:
if not srm_passed:
    conclusion = (
        "Investigate assignment imbalance before interpreting the result."
    )
elif p_value < ALPHA and lower_difference > 0:
    conclusion = (
        "The simulation provides evidence of higher treatment conversion."
    )
elif p_value < ALPHA and upper_difference < 0:
    conclusion = (
        "The simulation provides evidence of lower treatment conversion."
    )
else:
    conclusion = (
        "The simulation is inconclusive at the chosen significance level."
    )

print(conclusion)

print(
    f"\nObserved absolute lift: {difference * 100:.3f} percentage points"
)
print(
    f"Approximate 95% interval: "
    f"{lower_difference * 100:.3f} to "
    f"{upper_difference * 100:.3f} percentage points"
)
print(f"P-value: {p_value:.6f}")

print(
    "\nThis is a synthetic learning exercise. "
    "A real rollout decision also requires practical value, "
    "guardrail metrics, and trustworthy experiment execution."
)

The simulation provides evidence of higher treatment conversion.

Observed absolute lift: 0.213 percentage points
Approximate 95% interval: 0.074 to 0.351 percentage points
P-value: 0.002632

This is a synthetic learning exercise. A real rollout decision also requires practical value, guardrail metrics, and trustworthy experiment execution.
